<a href="https://colab.research.google.com/github/Karuneshtiwari/ML-lab/blob/main/Lab06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#A1- ChatGPT

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import time
import numpy as np
import pandas as pd

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

#A1 - ChatGPT

BASE_PATH = "/content/drive/MyDrive/ML_lab"
IMAGE_ROOT = os.path.join(BASE_PATH, "BreaKHis_v1")

CSV_PATH = os.path.join(
    BASE_PATH,
    "BreaKHis_metadata.csv"
)

SAMPLE_COUNT = 100
RESIZE_SHAPE = (16, 16)

DEFAULT_K = 3
DISTANCE = "euclidean"
SORT_METHOD = "insertion"

SEED = 42

Mounted at /content/drive


In [2]:
#A1/1 - ChatGPT

def locate_image(relative_name):
    relative_name = str(relative_name).replace("\\", "/")

    candidates = [
        os.path.join(BASE_PATH, relative_name),
        os.path.join(IMAGE_ROOT, relative_name)
    ]

    if relative_name.startswith("BreaKHis_v1/"):
        short_name = relative_name.split(
            "BreaKHis_v1/", 1
        )[1]

        candidates.append(
            os.path.join(IMAGE_ROOT, short_name)
        )

    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate

    return None


def get_target(path):
    path = str(path).lower()

    if "/benign/" in path:
        return "benign"

    if "/malignant/" in path:
        return "malignant"

    return None


def image_to_vector(path, size=(16, 16)):
    image = Image.open(path).convert("L")
    image = image.resize(size)

    values = np.asarray(
        image,
        dtype=np.float32
    )

    return values.reshape(-1)


def create_image_features(metadata, per_class=100):
    metadata = metadata.copy()

    metadata["target"] = metadata["filename"].map(
        get_target
    )

    metadata = metadata.dropna(
        subset=["target"]
    )

    selected_parts = []

    for target in ["benign", "malignant"]:

        class_rows = metadata[
            metadata["target"] == target
        ]

        selected_parts.append(
            class_rows.sample(
                n=min(per_class, len(class_rows)),
                random_state=SEED
            )
        )

    selected = pd.concat(
        selected_parts,
        ignore_index=True
    )

    features = []
    targets = []

    for _, record in selected.iterrows():

        image_path = locate_image(
            record["filename"]
        )

        if image_path is None:
            continue

        try:
            vector = image_to_vector(
                image_path,
                RESIZE_SHAPE
            )

            features.append(vector)
            targets.append(record["target"])

        except Exception:
            continue

    return (
        np.asarray(features, dtype=np.float32),
        np.asarray(targets)
    )

In [3]:
#A1/2 - ChatGPT

def encode_features(X):
    X = X.copy()

    for column in X.columns:

        if X[column].dtype == "object":

            values = X[column].dropna().unique()

            conversion = {
                value: position
                for position, value in enumerate(values)
            }

            X[column] = X[column].map(conversion)

    return X


def fill_missing(X, strategy="mean"):
    X = X.copy()

    for column in X.columns:

        if X[column].isna().any():

            if strategy == "mean":
                replacement = X[column].mean()

            elif strategy == "median":
                replacement = X[column].median()

            elif strategy == "mode":
                replacement = X[column].mode().iloc[0]

            else:
                raise ValueError(
                    "Use mean, median or mode."
                )

            X[column] = X[column].fillna(
                replacement
            )

    return X

In [4]:
#A1/3 - ChatGPT

def measure_distance(a, b, method="euclidean"):

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    difference = a - b

    if method == "euclidean":
        return np.linalg.norm(difference)

    if method == "manhattan":
        return np.sum(np.abs(difference))

    raise ValueError(
        "Unsupported distance method."
    )

In [5]:
#A1/4- ChatGPT

def bubble_order(records):

    output = records.copy()

    for end in range(
        len(output) - 1,
        0,
        -1
    ):

        for current in range(end):

            if (
                output[current][0],
                output[current][2]
            ) > (
                output[current + 1][0],
                output[current + 1][2]
            ):

                output[current], output[current + 1] = (
                    output[current + 1],
                    output[current]
                )

    return output


    #A1 - ChatGPT

def selection_order(records):

    output = records.copy()

    for position in range(len(output)):

        best = position

        for candidate in range(
            position + 1,
            len(output)
        ):

            if (
                output[candidate][0],
                output[candidate][2]
            ) < (
                output[best][0],
                output[best][2]
            ):

                best = candidate

        if best != position:

            output[position], output[best] = (
                output[best],
                output[position]
            )

    return output


    #A1 - ChatGPT

def insertion_order(records):

    output = records.copy()

    for position in range(1, len(output)):

        item = output[position]

        cursor = position - 1

        while cursor >= 0:

            previous = (
                output[cursor][0],
                output[cursor][2]
            )

            current = (
                item[0],
                item[2]
            )

            if previous <= current:
                break

            output[cursor + 1] = output[cursor]

            cursor -= 1

        output[cursor + 1] = item

    return output




In [6]:
#A1/5- ChatGPT

def order_neighbors(records, method):

    sorters = {
        "bubble": bubble_order,
        "selection": selection_order,
        "insertion": insertion_order
    }

    if method not in sorters:
        raise ValueError(
            "Invalid sorting method."
        )

    return sorters[method](records)

In [7]:
#A1 - ChatGPT

def nearest_items(sorted_records, k):

    if k < 1:
        raise ValueError(
            "k must be positive."
        )

    if k > len(sorted_records):
        raise ValueError(
            "k cannot exceed training samples."
        )

    return sorted_records[:k]


def majority_class(neighbors):

    votes = {}

    for distance, label, index in neighbors:

        votes[label] = votes.get(
            label,
            0
        ) + 1

    highest = max(votes.values())

    winners = [
        label
        for label, count in votes.items()
        if count == highest
    ]

    if len(winners) == 1:
        return winners[0]

    for _, label, _ in neighbors:

        if label in winners:
            return label

In [8]:
#A1 - ChatGPT

def predict_custom_knn(
    X_train,
    y_train,
    X_test,
    k=3,
    distance_method="euclidean",
    sorting_method="insertion"
):

    predictions = []

    for sample in X_test:

        distances = []

        for row_id, train_sample in enumerate(X_train):

            distance = measure_distance(
                sample,
                train_sample,
                distance_method
            )

            distances.append(
                (
                    distance,
                    y_train[row_id],
                    row_id
                )
            )

        ordered = order_neighbors(
            distances,
            sorting_method
        )

        selected = nearest_items(
            ordered,
            k
        )

        predictions.append(
            majority_class(selected)
        )

    return np.asarray(predictions)

In [9]:
#A1/A2 - ChatGPT

def weighted_class(neighbors):

    scores = {}

    for distance, label, index in neighbors:

        weight = 1.0 / max(
            distance,
            1e-10
        )

        scores[label] = (
            scores.get(label, 0.0)
            + weight
        )

    return max(
        scores,
        key=scores.get
    )


def predict_weighted_knn(
    X_train,
    y_train,
    X_test,
    k=3,
    distance_method="euclidean",
    sorting_method="insertion"
):

    predictions = []

    for sample in X_test:

        distances = []

        for row_id, train_sample in enumerate(X_train):

            distance = measure_distance(
                sample,
                train_sample,
                distance_method
            )

            distances.append(
                (
                    distance,
                    y_train[row_id],
                    row_id
                )
            )

        ordered = order_neighbors(
            distances,
            sorting_method
        )

        selected = nearest_items(
            ordered,
            k
        )

        predictions.append(
            weighted_class(selected)
        )

    return np.asarray(predictions)

In [10]:
#A1/A3 - ChatGPT

def make_split(X, y, test_ratio=0.20):

    return train_test_split(
        X,
        y,
        test_size=test_ratio,
        random_state=SEED,
        stratify=y
    )

In [11]:
#A1/A4 - ChatGPT

def sklearn_classifier(
    X_train,
    y_train,
    k=3
):

    classifier = KNeighborsClassifier(
        n_neighbors=k,
        metric=DISTANCE
    )

    classifier.fit(
        X_train,
        y_train
    )

    return classifier

In [12]:
#A1/A5/A6 - ChatGPT

def get_accuracy(actual, predicted):

    return accuracy_score(
        actual,
        predicted
    )


def make_prediction_table(
    actual,
    predicted
):

    table = pd.DataFrame({
        "Actual": actual,
        "Predicted": predicted
    })

    table["Correct"] = (
        table["Actual"]
        == table["Predicted"]
    )

    return table

In [13]:
#A1/A7 - ChatGPT

class ImageKNN:

    def __init__(
        self,
        neighbors=3,
        metric="euclidean",
        sorting="insertion"
    ):

        self.neighbors = neighbors
        self.metric = metric
        self.sorting = sorting

        self.features = None
        self.labels = None

    def fit(self, X, y):

        self.features = np.asarray(
            X,
            dtype=float
        )

        self.labels = np.asarray(y)

        return self

    def predict(self, X):

        return predict_custom_knn(
            self.features,
            self.labels,
            np.asarray(X, dtype=float),
            self.neighbors,
            self.metric,
            self.sorting
        )

    def score(self, X, y):

        predictions = self.predict(X)

        return accuracy_score(
            y,
            predictions
        )

In [14]:
#A1/A8 - ChatGPT

def evaluate_k_values(
    X_train,
    y_train,
    X_test,
    y_test,
    values
):

    records = []

    for k in values:

        custom_pred = predict_custom_knn(
            X_train,
            y_train,
            X_test,
            k,
            DISTANCE,
            SORT_METHOD
        )

        library_model = sklearn_classifier(
            X_train,
            y_train,
            k
        )

        library_pred = library_model.predict(
            X_test
        )

        records.append({
            "k": k,
            "Custom": get_accuracy(
                y_test,
                custom_pred
            ),
            "Scikit-learn": get_accuracy(
                y_test,
                library_pred
            )
        })

    return pd.DataFrame(records)

In [15]:
#A1/A9 - ChatGPT

def evaluate_weighting(
    X_train,
    y_train,
    X_test,
    y_test,
    values
):

    records = []

    for k in values:

        normal = predict_custom_knn(
            X_train,
            y_train,
            X_test,
            k,
            DISTANCE,
            SORT_METHOD
        )

        weighted = predict_weighted_knn(
            X_train,
            y_train,
            X_test,
            k,
            DISTANCE,
            SORT_METHOD
        )

        records.append({
            "k": k,
            "Normal kNN": get_accuracy(
                y_test,
                normal
            ),
            "Weighted kNN": get_accuracy(
                y_test,
                weighted
            )
        })

    return pd.DataFrame(records)

In [16]:
#main

metadata = pd.read_csv(CSV_PATH)

X, y = create_image_features(
    metadata,
    per_class=SAMPLE_COUNT
)

print("Dataset Shape:", X.shape)
print("\nClass Distribution:")
print(pd.Series(y).value_counts())


#A1

X_train_demo, X_test_demo, y_train_demo, y_test_demo = make_split(
    X,
    y
)

single_prediction = predict_custom_knn(
    X_train_demo,
    y_train_demo,
    X_test_demo[:1],
    DEFAULT_K,
    DISTANCE,
    SORT_METHOD
)

print("\nA1")
print("Actual Class:", y_test_demo[0])
print("Predicted Class:", single_prediction[0])


#A2

weighted_prediction = predict_weighted_knn(
    X_train_demo,
    y_train_demo,
    X_test_demo[:1],
    DEFAULT_K,
    DISTANCE,
    SORT_METHOD
)

print("\nA2")
print("Actual Class:", y_test_demo[0])
print(
    "Weighted Prediction:",
    weighted_prediction[0]
)


#A3

print("\nA3")
print("Training Samples:", len(X_train_demo))
print("Testing Samples:", len(X_test_demo))


#A4

model = sklearn_classifier(
    X_train_demo,
    y_train_demo,
    DEFAULT_K
)

sklearn_predictions = model.predict(
    X_test_demo
)

print("\nA4")
print(
    "Scikit-learn kNN completed for k =",
    DEFAULT_K
)


#A5

accuracy = get_accuracy(
    y_test_demo,
    sklearn_predictions
)

print("\nA5")
print(
    "Scikit-learn Accuracy:",
    round(accuracy, 4)
)


#A6

prediction_results = make_prediction_table(
    y_test_demo,
    sklearn_predictions
)

print("\nA6")
display(
    prediction_results.head(10)
)


#A7

custom_model = ImageKNN(
    neighbors=DEFAULT_K,
    metric=DISTANCE,
    sorting=SORT_METHOD
)

custom_model.fit(
    X_train_demo,
    y_train_demo
)

custom_accuracy = custom_model.score(
    X_test_demo,
    y_test_demo
)

print("\nA7")
print(
    "Custom kNN Accuracy:",
    round(custom_accuracy, 4)
)


#A8

k_options = [1, 3, 5, 7, 9]

k_results = evaluate_k_values(
    X_train_demo,
    y_train_demo,
    X_test_demo,
    y_test_demo,
    k_options
)

print("\nA8")
display(k_results)


#A9

weight_results = evaluate_weighting(
    X_train_demo,
    y_train_demo,
    X_test_demo,
    y_test_demo,
    k_options
)

print("\nA9")
display(weight_results)

Dataset Shape: (200, 256)

Class Distribution:
benign       100
malignant    100
Name: count, dtype: int64

A1
Actual Class: benign
Predicted Class: benign

A2
Actual Class: benign
Weighted Prediction: benign

A3
Training Samples: 160
Testing Samples: 40

A4
Scikit-learn kNN completed for k = 3

A5
Scikit-learn Accuracy: 0.525

A6


,Actual,Predicted,Correct
0,benign,benign,True
1,malignant,malignant,True
2,malignant,malignant,True
3,malignant,malignant,True
4,benign,malignant,False
5,malignant,benign,False
6,benign,benign,True
7,malignant,malignant,True
8,malignant,malignant,True
9,benign,malignant,False



A7
Custom kNN Accuracy: 0.525

A8


,k,Custom,Scikit-learn
0,1,0.525,0.525
1,3,0.525,0.525
2,5,0.475,0.475
3,7,0.525,0.525
4,9,0.550,0.550



A9


,k,Normal kNN,Weighted kNN
0,1,0.525,0.525
1,3,0.525,0.525
2,5,0.475,0.475
3,7,0.525,0.525
4,9,0.550,0.550
